In [0]:
from pyspark.sql.functions import col, trim, initcap, lower, lit, when

# Create messy raw dataset
raw_data = [
    (101, "  ALICE smith ", " ALICE@EMAIL.COM", "NY", "Y"),
    (102, "bob jones", None, "CA", "Y"), 
    (103, "CHARLIE  ", "charlie@email.com", "TX", "Y"),
    (103, "CHARLIE  ", "charlie@email.com", "TX", "Y"), # Duplicate
    (104, "  ", "david@email.com", "FL", "Y"), # Missing string
    (105, "Eve Davis", "eve@email.com", None, "Y") # Null
]
columns = ["customer_id", "name", "email", "state", "is_active"]

df_raw = spark.createDataFrame(raw_data, columns)
df_raw.write.format("delta").mode("overwrite").saveAsTable("customer_bronze")

# Take a screenshot of this output
display(spark.read.table("customer_bronze"))

customer_id,name,email,state,is_active
101,ALICE smith,ALICE@EMAIL.COM,NY,Y
102,bob jones,null,CA,Y
103,CHARLIE,charlie@email.com,TX,Y
103,CHARLIE,charlie@email.com,TX,Y
104,,david@email.com,FL,Y
105,Eve Davis,eve@email.com,null,Y


In [0]:
# Read Bronze and clean
df_bronze = spark.read.table("customer_bronze")

df_silver = (df_bronze
    .dropDuplicates()
    .withColumn("name", initcap(trim(col("name"))))
    .withColumn("name", when(col("name") == "", lit("Unknown")).otherwise(col("name")))
    .withColumn("email", lower(trim(col("email"))))
    .fillna({"email": "no-email@provided.com", "state": "UNKNOWN"})
)

df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("customer_silver")

# Take a screenshot of this output
display(spark.read.table("customer_silver"))

customer_id,name,email,state,is_active
104,Unknown,david@email.com,FL,Y
102,Bob Jones,no-email@provided.com,CA,Y
105,Eve Davis,eve@email.com,UNKNOWN,Y
101,Alice Smith,alice@email.com,NY,Y
103,Charlie,charlie@email.com,TX,Y


In [0]:
# Setup cell: Create new day's data
inc_data = [
    (101, "Alice Smith-Johnson", "alice.new@email.com", "NY", "Y"), 
    (106, "Frank Wright", "frank@email.com", "WA", "Y"),          
    (103, "Charlie", "charlie@email.com", "TX", "Y")              
]
spark.createDataFrame(inc_data, columns).createOrReplaceTempView("customer_incremental_view")
display(df_inc)

customer_id,name,email,state,is_active
101,Alice Smith-Johnson,alice.new@email.com,NY,Y
106,Frank Wright,frank@email.com,WA,Y
103,Charlie,charlie@email.com,TX,Y


In [0]:
%sql
MERGE INTO customer_silver AS target
USING customer_incremental_view AS source
ON target.customer_id = source.customer_id

-- SCD1 Logic: Update existing or Insert new
WHEN MATCHED THEN
  UPDATE SET target.name = source.name, target.email = source.email
WHEN NOT MATCHED THEN
  INSERT *
  
-- Soft Delete Logic
WHEN NOT MATCHED BY SOURCE THEN
  UPDATE SET target.is_active = 'N'

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
6,5,0,1


In [0]:
# Validate the Soft Delete (Records missing from source are flagged, not deleted)
df_soft_deleted = spark.read.table("customer_silver").filter(col("is_active") == 'N')

# Take a screenshot of this output showing the 'N' records
display(df_soft_deleted)

customer_id,name,email,state,is_active
104,Unknown,david@email.com,FL,N
102,Bob Jones,no-email@provided.com,CA,N
105,Eve Davis,eve@email.com,UNKNOWN,N


In [0]:
# Validate the active dataset for downstream analytics
df_active_customers = spark.read.table("customer_silver").filter(col("is_active") == 'Y').orderBy("customer_id")

# Take a screenshot of this output
print(f"Total Active Customers: {df_active_customers.count()}")
display(df_active_customers)

Total Active Customers: 3


customer_id,name,email,state,is_active
101,Alice Smith-Johnson,alice.new@email.com,NY,Y
103,Charlie,charlie@email.com,TX,Y
106,Frank Wright,frank@email.com,WA,Y


In [0]:
# Display the entire final dataset (both active and inactive records)
df_final = spark.read.table("customer_silver").orderBy("customer_id")

# Take a screenshot of this output
display(df_final)

customer_id,name,email,state,is_active
101,Alice Smith-Johnson,alice.new@email.com,NY,Y
102,Bob Jones,no-email@provided.com,CA,N
103,Charlie,charlie@email.com,TX,Y
104,Unknown,david@email.com,FL,N
105,Eve Davis,eve@email.com,UNKNOWN,N
106,Frank Wright,frank@email.com,WA,Y
